<a href="https://colab.research.google.com/github/anaclaracandidoo/projetodeep-learning/blob/main/10_projeto_unificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf projetodeep-learning
!git clone https://github.com/anaclaracandidoo/projetodeep-learning.git
%cd projetodeep-learning
!pip install -q -r requirements.txt

In [ ]:
import os
import asyncio
import uvicorn
from pyngrok import ngrok
from main import app

# Opcional: coloque sua chave gratuita do Google AI Studio aqui se quiser usar IA generativa na adaptação
os.environ["GEMINI_API_KEY"] = ""

tunnel = ngrok.connect(8000)
URL_API_ADAPTAR = f"{tunnel.public_url}/file/adaptar"
print("API rodando em:", URL_API_ADAPTAR)

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
asyncio.create_task(server.serve())

In [ ]:
import gradio as gr
import requests
from datetime import datetime

historico_materiais = []

def processar_arquivo(arquivo):
    if arquivo is None:
        return "Selecione um arquivo (PDF, DOCX ou TXT).", "", None, historico_materiais

    try:
        with open(arquivo.name, "rb") as f:
            files = {"file": (arquivo.name, f)}
            headers = {"ngrok-skip-browser-warning": "true"}
            response = requests.post(URL_API_ADAPTAR, files=files, headers=headers)

        if response.status_code == 200:
            dados = response.json()
            nome = dados.get("filename", "arquivo").split("/")[-1]
            orig = dados.get("original_text", "")
            adap = dados.get("adapted_text", "")
            qtd_orig = dados.get("char_count_original", len(orig))
            qtd_adap = dados.get("char_count_adapted", len(adap))

            caminho_txt = f"/tmp/ADAPTADO_{nome}.txt"
            with open(caminho_txt, "w", encoding="utf-8") as out:
                out.write(adap)

            historico_materiais.append([
                datetime.now().strftime("%d/%m/%Y %H:%M"),
                nome,
                f"{qtd_orig} caracteres",
                f"{qtd_adap} caracteres"
            ])

            return f"📄 {nome} ({qtd_orig} caracteres)\n\n{orig}", adap, caminho_txt, historico_materiais
        else:
            return f"Erro ({response.status_code}): {response.text}", "", None, historico_materiais
    except Exception as e:
        return f"Erro de conexão: {str(e)}", "", None, historico_materiais

with gr.Blocks(theme=gr.themes.Soft(), title="EduAcessível") as demo:
    gr.Markdown("# 🎓🤟 EduAcessível — Painel da Professora")
    gr.Markdown("Adaptação automática de materiais didáticos para alunos surdos (Português L2 / Estrutura Visual).")

    with gr.Tabs():
        with gr.TabItem("🏠 Dashboard"):
            arq_in = gr.File(label="Envie o plano de aula ou atividade (PDF, DOCX ou TXT)", file_types=[".pdf", ".docx", ".txt"])
            btn = gr.Button("🚀 Extrair e Adaptar Material", variant="primary")
            with gr.Row():
                txt_orig = gr.Textbox(label="📄 Texto Original Extraído", lines=16)
                txt_adap = gr.Textbox(label="🤟 Material Adaptado para Alunos Surdos", lines=16)
            arq_out = gr.File(label="⬇️ Baixar Material Adaptado (.TXT)")

        with gr.TabItem("🕒 Histórico de Materiais"):
            tab_hist = gr.Dataframe(
                headers=["Data/Hora", "Arquivo", "Original", "Adaptado"],
                label="Materiais adaptados nesta sessão"
            )

    btn.click(
        fn=processar_arquivo,
        inputs=[arq_in],
        outputs=[txt_orig, txt_adap, arq_out, tab_hist]
    )

demo.launch(share=True)